In [158]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, accuracy_score
import os
from tensorflow.keras import layers, applications

In [159]:
MODEL1_DIR = '/kaggle/input/densenet121-ensemble/DenseNet121_Ensemble'
preprocess_fn1 = applications.densenet.preprocess_input

MODEL2_DIR = '/kaggle/input/resnet50-ensemble/ResNet50_Ensemble'
preprocess_fn2 = applications.resnet.preprocess_input

MODEL3_DIR = '/kaggle/input/vgg19-ensemble/VGG19_Ensemble'
preprocess_fn3 = applications.vgg19.preprocess_input

MODEL4_DIR = '/kaggle/input/xception-ensemble/Xception_Ensemble'
preprocess_fn4 = applications.xception.preprocess_input

MODEL5_DIR = '/kaggle/input/mobilenet-ensemble/MobileNet_Ensemble'
preprocess_fn5 = applications.mobilenet.preprocess_input

In [160]:
TRAIN_DIR = '/kaggle/input/5-fold-brain-tumor-contrast-enhanced/kfold_dataset'

In [161]:
def build_data_augmentation(SEED=24520152):
    """Create a simple data augmentation pipeline"""
    return tf.keras.Sequential([
        layers.RandomFlip('horizontal', seed=SEED),
        layers.RandomRotation(0.1, seed=SEED),
        layers.RandomZoom(0.1, seed=SEED),
        layers.RandomContrast(0.1, seed=SEED),
        layers.RandomBrightness(0.1, seed=SEED),
    ], name='data_augmentation')

In [ ]:
def get_pred_for_ensembling_model(fold_k, val_ds):
    # model1 = tf.keras.models.load_model(os.path.join(MODEL1_DIR, f'DenseNet121_block_2_fold_{fold_k}.keras'), compile=False)
    model2 = tf.keras.models.load_model(os.path.join(MODEL2_DIR, f'ResNet50_block_1_fold_{fold_k}.keras'), compile=False)
    # model3 = tf.keras.models.load_model(os.path.join(MODEL3_DIR, f'VGG19_block_1_fold_{fold_k}.keras'), compile=False)
    model4 = tf.keras.models.load_model(os.path.join(MODEL4_DIR, f'Xception_block_2_fold_{fold_k}.keras'), compile=False)
    model5 = tf.keras.models.load_model(os.path.join(MODEL5_DIR, f'MobileNet_block_1_fold_{fold_k}.keras'), compile=False)

    AUTOTUNE = tf.data.AUTOTUNE
    # val_ds1 = val_ds.map(lambda image, label: (preprocess_fn1(image), label)).prefetch(buffer_size=AUTOTUNE)
    val_ds2 = val_ds.map(lambda image, label: (preprocess_fn2(image), label)).prefetch(buffer_size=AUTOTUNE)
    # val_ds3 = val_ds.map(lambda image, label: (preprocess_fn3(image), label)).prefetch(buffer_size=AUTOTUNE)
    val_ds4 = val_ds.map(lambda image, label: (preprocess_fn4(image), label)).prefetch(buffer_size=AUTOTUNE)
    val_ds5 = val_ds.map(lambda image, label: (preprocess_fn5(image), label)).prefetch(buffer_size=AUTOTUNE)

    y_true = np.concatenate([y.numpy() for _, y in val_ds], axis=0)
    # pred1 = model1.predict(val_ds1)
    pred2 = model2.predict(val_ds2)
    # pred3 = model3.predict(val_ds3)
    pred4 = model4.predict(val_ds4)
    pred5 = model5.predict(val_ds5)
    ensemble_pred = (pred2 + pred4 + pred5) / 3.0

    return ensemble_pred

In [163]:
DATASET_CACHE = {}
def get_validation_fold(k, train_dir=TRAIN_DIR, IMG_SIZE=(224, 224), BATCH_SIZE=32, SEED=24520152):
    val_dir = os.path.join(train_dir, f'Subset_{k}')
    
    val_ds = tf.keras.utils.image_dataset_from_directory(
        val_dir,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=False,
        seed=SEED,
    )
    
    AUTOTUNE = tf.data.AUTOTUNE

    return val_ds

In [164]:
def get_predictions_for_fold(k):
    valid_data = get_validation_fold(k)

    y_true = []
    for _, y in valid_data:
        y_true.extend(y.numpy())
    y_true = np.array(y_true)
    y_pred = []

    y_pred.append(get_pred_for_ensembling_model(k, valid_data))
    
    y_pred = np.array(y_pred)
    return y_true, y_pred

In [165]:
def macro_specificity(y_true, y_pred, num_classes):
    cm = confusion_matrix(y_true, y_pred)
    spec = []

    for i in range(num_classes):
        TP = cm[i, i]
        FN = cm[i, :].sum() - TP
        FP = cm[:, i].sum() - TP
        TN = cm.sum() - (TP + FN + FP)

        spec.append(TN / (TN + FP + 1e-8))

    return np.array(spec)

In [166]:
def evaluate_fold(k):
    y_true, y_pred = get_predictions_for_fold(k)

    model_results = []
    for i in range(len(y_pred)):
        y_pred_label = np.argmax(y_pred[i], axis=1)
        acc = accuracy_score(y_true, y_pred_label)
        precision = precision_score(y_true, y_pred_label, average='macro', zero_division=0)
        recall = recall_score(y_true, y_pred_label, average='macro', zero_division=0)
        f1 = f1_score(y_true, y_pred_label, average='macro', zero_division=0)
        spec_per_class = macro_specificity(y_true, y_pred_label, 3)
        specificity = np.mean(spec_per_class)

        model_results.append({
            'Accuracy': np.array(acc),
            'Precision': np.array(precision),
            'Recall': np.array(recall),
            'F1-Score': np.array(f1),
            'Specificity': np.array(specificity)
        })
    return np.array(model_results)

In [167]:
five_fold_results = []
for i in range(1, 6):
    five_fold_results.append(evaluate_fold(i))

five_fold_results = np.array(five_fold_results)

num_fold = 5
num_model = 1
metric_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'Specificity']

avg_metrics = {}

for m in range(num_model):
    avg_metrics[m] = {}
    for metric in metric_names:
        values = np.array([
            five_fold_results[k][m][metric] for k in range(num_fold)
        ])
        avg_metrics[m][metric] = values.mean(axis=0)*100

Found 542 files belonging to 3 classes.
17/17 ━━━━━━━━━━━━━━━━━━━━ 10s 283ms/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 305ms/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 5s 175ms/step
Found 679 files belonging to 3 classes.
22/22 ━━━━━━━━━━━━━━━━━━━━ 10s 274ms/step
22/22 ━━━━━━━━━━━━━━━━━━━━ 9s 248ms/step
22/22 ━━━━━━━━━━━━━━━━━━━━ 6s 153ms/step
Found 572 files belonging to 3 classes.
18/18 ━━━━━━━━━━━━━━━━━━━━ 9s 274ms/step
18/18 ━━━━━━━━━━━━━━━━━━━━ 11s 319ms/step
18/18 ━━━━━━━━━━━━━━━━━━━━ 6s 177ms/step
Found 628 files belonging to 3 classes.
20/20 ━━━━━━━━━━━━━━━━━━━━ 8s 238ms/step
20/20 ━━━━━━━━━━━━━━━━━━━━ 9s 263ms/step
20/20 ━━━━━━━━━━━━━━━━━━━━ 6s 194ms/step
Found 643 files belonging to 3 classes.
21/21 ━━━━━━━━━━━━━━━━━━━━ 9s 231ms/step
21/21 ━━━━━━━━━━━━━━━━━━━━ 9s 257ms/step
21/21 ━━━━━━━━━━━━━━━━━━━━ 5s 144ms/step


In [ ]:
row_names = ['ensemble-resnet50-xception-mobilenet']
column_names = ['Recall', 'Specificity', 'Precision', 'F1-Score', 'Accuracy']

df_result = pd.DataFrame(avg_metrics).T
df_result.index = row_names
df_result = df_result[column_names]
df_result = df_result.round(2)
df_result

,Recall,Specificity,Precision,F1-Score,Accuracy
ensemble-resnet50-xception-mobilenet,91.79,96.48,92.53,92.0,93.14


In [169]:
latex_table = df_result.to_latex(
    multicolumn=True,
    multirow=True,
    float_format="%.2f",
    label="tab:sens_spec"
)

print(latex_table)

\begin{table}
\label{tab:sens_spec}
\begin{tabular}{lrrrrr}
\toprule
 & Recall & Specificity & Precision & F1-Score & Accuracy \\
\midrule
ensemble-resnet50-xception-mobilenet & 91.79 & 96.48 & 92.53 & 92.00 & 93.14 \\
\bottomrule
\end{tabular}
\end{table}

